# 04 · Modelul de vulnerabilitate (LightGBM + SHAP)
Explică pierderea observată în 2024 (ținta din `03e`) prin factori structurali, doar pe parcelele arabile. Validare pe blocuri spațiale de 5 km, explicații cu SHAP pe fiecare parcelă.

Rezultatul arată **asocieri, nu cauze dovedite**. Harta-erou arată pierderea observată; scorul de aici e panou secundar, cu R² spațial scris pe slide.

In [ ]:
# Sincronizare: aduce repo-ul pe runtime și încarcă configurarea.
# Local (kernel pe PC) găsește direct ../gisday_config.py; pe Colab clonează repo-ul privat.
import os, sys, subprocess, base64, getpass

def _auth(tok):
    b = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return ['-c', f'http.extraheader=AUTHORIZATION: basic {b}']

def _token():
    global TOKEN
    if 'TOKEN' not in globals():
        TOKEN = getpass.getpass('Token GitHub (fine-grained, doar repo-ul gisday2026): ')
    return TOKEN

if not os.path.exists('gisday_config.py'):
    if os.path.exists('../gisday_config.py'):
        os.chdir('..')
    else:
        REPO = 'https://github.com/lefterpatrickandrei-sketch/gisday2026.git'   # același cu GITHUB_REPO din gisday_config.py
        if not os.path.exists('/content/gisday2026'):
            subprocess.run(['git', *_auth(_token()), 'clone', REPO, '/content/gisday2026'], check=True)
        os.chdir('/content/gisday2026')
sys.path.insert(0, os.getcwd())
from gisday_config import *

def pull():
    subprocess.run(['git', *_auth(_token()), 'pull', '--ff-only'], check=True)

def push(paths, msg):
    subprocess.run(['git', 'add', *[str(p) for p in paths]], check=True)
    subprocess.run(['git', '-c', 'user.name=gisday-runtime', '-c', 'user.email=gisday@users.noreply.github.com',
                    'commit', '-m', msg], check=False)
    subprocess.run(['git', *_auth(_token()), 'push'], check=True)

print('Lucrez în', os.getcwd())

In [ ]:
%pip install -q lightgbm shap geopandas pyarrow matplotlib

In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, lightgbm as lgb, shap, json
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error
parcels = gpd.read_parquet(OUT / 'parcels.parquet')
F = pd.read_parquet(OUT / 'parcel_features.parquet')
T = pd.read_parquet(OUT / 'parcel_target.parquet')          # ținta din 03e: pierderea față de anii cu același tip de cultură
D = parcels.merge(F, on='parcel_id').merge(T, on='parcel_id')
for name in ('parcel_canals', 'parcel_wind'):               # factorii din 03f și 03c
    if (OUT / f'{name}.parquet').exists():
        D = D.merge(pd.read_parquet(OUT / f'{name}.parquet'), on='parcel_id')
TARGET = D.loss_source.iloc[0]
D = D[D.is_arable & D.loss_obs.notna()].copy()               # doar parcele arabile cu țintă
D['loss'] = D['loss_obs']                                    # mai mare = pagubă mai mare
print('ținta:', TARGET, '| parcele:', len(D))
print(D['loss'].describe(percentiles=[.05, .25, .5, .75, .95]).round(3).to_string())

In [ ]:
# Factori structurali: DOAR coloanele care există. Fără indici din vara 2024 și fără irrigated_prob (ar fi circular),
# fără cultura din EUCROPMAP 2022 (nu spune ce s-a cultivat în 2024).
CANDIDATES = ['dist_canal_m', 'dist_network_m', 'wind_exposure', 'slope', 'elev',
              'soc_sg_top', 'sand_sg_top', 'clay_sg_top', 'soc_lucas_model', 'area_ha']
FEATURES = [c for c in CANDIDATES if c in D.columns]
print('factori:', FEATURES)
D = D.dropna(subset=['loss'] + FEATURES).copy()
c = D.geometry.centroid
D['block'] = (c.x // 5000).astype(int).astype(str) + '_' + (c.y // 5000).astype(int).astype(str)
X, y = D[FEATURES], D['loss']
print('parcele în model:', len(D), '| blocuri de 5 km:', D['block'].nunique())

In [ ]:
params = dict(n_estimators=600, learning_rate=0.03, num_leaves=31, min_child_samples=30,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
pred = np.zeros(len(D))
for k, (tr, te) in enumerate(GroupKFold(n_splits=5).split(X, y, D['block'])):
    m = lgb.LGBMRegressor(**params).fit(X.iloc[tr], y.iloc[tr])
    pred[te] = m.predict(X.iloc[te])
    print(f'fold {k}: R2={r2_score(y.iloc[te], pred[te]):.3f}')
metrics = dict(target=TARGET, r2_spatial=round(float(r2_score(y, pred)), 3), rmse=round(float(mean_squared_error(y, pred) ** 0.5), 4),
               n_parcels=len(D), n_blocks=int(D['block'].nunique()), factors=FEATURES)
print('validare spațială:', metrics)
(OUT / 'model_metrics.json').write_text(json.dumps(metrics, indent=1), encoding='utf-8')

In [ ]:
model = lgb.LGBMRegressor(**params).fit(X, y)
sv = shap.TreeExplainer(model).shap_values(X)
S = pd.DataFrame(sv, columns=[f'shap_{f}' for f in FEATURES], index=D.index)
D['pred_loss'] = model.predict(X)
D['vuln_score'] = (D['pred_loss'].rank(pct=True) * 100).round(1)       # 0–100
D['driver'] = S.idxmax(axis=1).str.replace('shap_', '', regex=False)   # factorul care împinge cel mai mult spre pagubă
D = pd.concat([D, S], axis=1)
cols = ['parcel_id', 'area_ha', 'loss', 'pred_loss', 'vuln_score', 'driver'] + list(S.columns) + ['geometry']
R = gpd.GeoDataFrame(D[cols], geometry='geometry', crs=CRS)
R.to_parquet(OUT / 'vulnerability.parquet')
R.to_file(OUT / 'vulnerability.gpkg', layer='vulnerabilitate', driver='GPKG')
print(R['driver'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
imp = S.abs().mean().sort_values()
imp.index = imp.index.str.replace('shap_', '', regex=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp.index, imp.values, color='#3987e5')
ax.set_xlabel('|SHAP| mediu (contribuție la pagubă)')
ax.set_title('Ce explică vulnerabilitatea în 2024')
plt.tight_layout(); plt.savefig(OUT / 'shap_importance.png', dpi=200); plt.show()

In [ ]:
if os.path.exists('/content'):     # pe PC comiți normal, din git
    push([OUT / 'vulnerability.parquet', OUT / 'vulnerability.gpkg', OUT / 'shap_importance.png'], 'model vulnerabilitate')